# NDJF Gap-Merge Catalog Cleanup

This notebook builds merged candidate catalogs from the existing NDJF event table without rerunning the expensive ERA5 detection workflow.

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = "main"
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = True
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )
else:
    print("Using existing repo clone:", REPO_DIR)

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("Working directory:", os.getcwd())


In [ ]:
from pathlib import Path
import shutil

import pandas as pd

from jpcz_catalog.detect import merge_gap_sensitivity, merge_nearby_events
from jpcz_catalog.verification import render_gap_merge_summary, write_text_summary

CATALOG_PATH = Path("outputs/verification/jpcz_catalog_ndjf.csv")
MERGE_GAP_VALUES = (6, 12, 24)
RECOMMENDED_GAP_HOURS = 12

catalog_df = pd.read_csv(
    CATALOG_PATH,
    parse_dates=["event_start", "event_end", "event_peak"],
)

sensitivity_df = merge_gap_sensitivity(
    catalog_df,
    gap_hours_values=MERGE_GAP_VALUES,
)
sensitivity_path = Path("outputs/verification/jpcz_catalog_ndjf_gap_merge_sensitivity.csv")
sensitivity_df.to_csv(sensitivity_path, index=False)

merged_catalog = merge_nearby_events(
    catalog_df,
    max_gap_hours=RECOMMENDED_GAP_HOURS,
)
merged_catalog_path = Path(f"outputs/verification/jpcz_catalog_ndjf_merged_{int(RECOMMENDED_GAP_HOURS)}h.csv")
merged_catalog.to_csv(merged_catalog_path, index=False)

summary_text = render_gap_merge_summary(
    sensitivity_df=sensitivity_df,
    recommended_gap_hours=RECOMMENDED_GAP_HOURS,
    merged_catalog_name=merged_catalog_path.name,
)
summary_path = write_text_summary(
    f"outputs/verification/jpcz_catalog_ndjf_merged_{int(RECOMMENDED_GAP_HOURS)}h_summary.md",
    summary_text,
)

if PERSIST_OUTPUTS_TO_DRIVE:
    for path in [sensitivity_path, merged_catalog_path, summary_path]:
        shutil.copy2(path, Path(DRIVE_OUTPUT_DIR) / path.name)

print(summary_text)
merged_catalog.head()


In [ ]:
EXAMPLE_DATES = [
    "2002-11-03",
    "2002-01-21",
    "2004-01-21",
]

for target in EXAMPLE_DATES:
    print(f"\nTARGET {target} | raw catalog")
    display(
        catalog_df[
            catalog_df["event_start"].astype(str).str.startswith(target)
            | catalog_df["event_end"].astype(str).str.startswith(target)
            | catalog_df["event_peak"].astype(str).str.startswith(target)
        ][["event_start", "event_end", "event_peak", "duration_hours", "event_peak_D_1e5_s-1"]]
    )

    print(f"TARGET {target} | merged catalog")
    display(
        merged_catalog[
            merged_catalog["event_start"].astype(str).str.startswith(target)
            | merged_catalog["event_end"].astype(str).str.startswith(target)
            | merged_catalog["event_peak"].astype(str).str.startswith(target)
        ][[
            "event_start",
            "event_end",
            "event_peak",
            "duration_hours",
            "threshold_hit_hours",
            "merged_subevents_count",
            "max_internal_gap_hours",
            "event_peak_D_1e5_s-1",
        ]]
    )
